# 04. Feature Engineering
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the philosophy of **Feature Engineering**: transforming raw data into informative signals that linear and non-linear algorithms can exploit.
2. Apply **continuous feature transformations**: log transforms (`np.log1p`) and quantile-based binning (`pd.cut`, `pd.qcut`).
3. Master all core **categorical encoding techniques**:
   - One-Hot Encoding (`OneHotEncoder`)
   - Ordinal & Label Encoding
   - Frequency / Count Encoding
   - Out-of-fold Target / Mean Encoding (and how to avoid catastrophic target leakage)
4. Engineer **domain-specific interaction features** (ratios, bundled service counts).
5. Apply **feature scaling** (`StandardScaler`, `RobustScaler`) strictly within the leak-free train/test discipline.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    RobustScaler,
)
from sklearn.model_selection import KFold

import warnings

warnings.filterwarnings("ignore")

# Load the cleaned partitions prepared in Notebook 03
X_train = pd.read_csv("../data/telco_X_train_clean.csv")
X_test = pd.read_csv("../data/telco_X_test_clean.csv")
y_train = pd.read_csv("../data/telco_y_train.csv")["Churn"]
y_test = pd.read_csv("../data/telco_y_test.csv")["Churn"]

# Map target to binary integer (1 = Churn, 0 = Retained)
y_train_num = (y_train == "Yes").astype(int)
y_test_num = (y_test == "Yes").astype(int)

print(f"Loaded Cleaned Train: {X_train.shape}, Test: {X_test.shape}")
X_train.head(3)

## 1. Continuous Feature Transformations

Raw continuous features often have heavily right-skewed distributions or non-linear step thresholds.

### A. Log Transformation
When a positive continuous feature has high skew (e.g. `TotalCharges`), taking $\log(1 + x)$ compresses extreme values and brings the distribution closer to normal, which dramatically stabilizes linear and distance-based algorithms.


In [ ]:
# TODO: Apply log1p transformation to TotalCharges to reduce skewness
# YOUR CODE HERE


### B. Discretization / Binning (`pd.cut` & `pd.qcut`)
Sometimes business rules or customer behavior change at specific lifecycle milestones.
- **`pd.cut`**: Bins into intervals of equal width or custom domain edges.
- **`pd.qcut`**: Bins into quantiles with an equal number of samples per bin.


In [ ]:
# TODO: Discretize 'tenure' into domain cohorts using pd.cut:
# Bins: [-1, 12, 24, 48, 73]
# Labels: ['New (0-12m)', 'Onboarding (1-2y)', 'Established (2-4y)', 'Loyal (4-6y)']
# YOUR CODE HERE


## 2. Domain-Specific Interaction Features

Senior data scientists consult domain experts to build features that capture customer behavior directly:

1. **Service Stickiness Count**: A customer with 6 bundled add-ons (Security, Backup, Streaming, TechSupport) is significantly less likely to churn than a customer with only raw internet.
2. **Monthly Charge to Tenure Ratio**: Captures financial intensity relative to customer loyalty.


In [ ]:
# TODO: Engineer domain features:
# 1. total_services: Count of active services
# 2. charge_tenure_ratio: MonthlyCharges / (tenure + 1)
# 3. is_auto_payment: 1 if automatic payment method, else 0
# YOUR CODE HERE


## 3. Categorical Encodings

Machine learning models operate on matrices of numbers. We must convert categorical strings into numerical representations without introducing false ordering.

### A. One-Hot Encoding (Nominal Features)
- For nominal categories with no natural rank (`Contract`, `InternetService`, `PaymentMethod`).
- Creates $K-1$ binary dummy columns (`drop='first'`) to prevent multicollinearity.


In [ ]:
# TODO: One-Hot Encode nominal categorical columns using OneHotEncoder(drop='first', sparse_output=False)
# Fit on Train, transform Train and Test
# YOUR CODE HERE


In [ ]:
# Inspect encoded columns
# YOUR CODE HERE


### B. Frequency / Count Encoding
Replaces each category with its relative frequency (percentage of the dataset). Highly effective for categories with moderate-to-high cardinality.


In [ ]:
# TODO: Compute frequency encoding for PaymentMethod on Train and map to Train and Test
# YOUR CODE HERE


### C. Target / Mean Encoding (with Out-of-Fold Cross-Validation)

Target encoding replaces a category with the average target value ($P(y=1 | \text{Category})$) for that category.

> ⚠️ **THE TARGET LEAKAGE TRAP**:
> If you compute target means on the full training set without cross-validation, a rare category with 1 sample that churned receives an encoded value of $1.00$. The model will overfit completely!
> **Solution**: Use **Out-of-Fold (OOF)** K-Fold encoding during training, and overall training means on the test set!


In [ ]:
store1 = pd.DataFrame(
    {"store": ["A"] * 3, "Sales": [100, 200, 300], "noise": [0, 0, 0]}
)

store2 = pd.DataFrame({"store": ["B"] * 3, "Sales": [10, 20, 30], "noise": [0, 0, 0]})

data = pd.concat([store1, store2], axis=0)
data

In [ ]:
def mean_encode(data, col, on):
    data = data.copy()

    # Compute mean target per category
    means = data.groupby(col)[on].mean()

    # Preserve original column
    data[f"{col}-original"] = data[col]

    # Apply encoding
    data[col] = data[col].map(means)

    # Fill unseen categories with global mean
    data[col] = data[col].fillna(data[on].mean())

    return data


# define data
store1 = pd.DataFrame(
    {"store": ["A"] * 3, "Sales": [100, 200, 300], "noise": [0, 0, 0]}
)

store2 = pd.DataFrame({"store": ["B"] * 3, "Sales": [10, 20, 30], "noise": [0, 0, 0]})

data = pd.concat([store1, store2], axis=0)
data = mean_encode(data, col="store", on="Sales")

data

In [ ]:
# TODO: Implement out-of-fold target encoding with KFold to prevent target leakage
# Apply to 'Contract' feature
# YOUR CODE HERE


## C. Binary Encoding
Encoding the string using it's binary (0110100 etc) representation

In Python we can do this using the `bin()` builtin:

In [ ]:
# Number 2 in binary as a string
bin(2)

In [ ]:
bin(16)

In [ ]:
import category_encoders as ce

data = ["A", "BB", "B", "A", "C", "D", "A", "E"]
# data = [1,2,3,1,4,5,1,6]

enc = (
    ce.BinaryEncoder()
)  # similar to onehot, but stores categories as binary bitstrings.

print("{} classes".format(len(set(data))))

pd.concat([enc.fit_transform(data), pd.DataFrame(data, columns=["data"])], axis=1)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

data_df = pd.DataFrame({"data": data})
encoded = ohe.fit_transform(data_df)

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(["data"]))

pd.concat([encoded_df, data_df], axis=1)

## 4. Feature Scaling (Fit on Train, Transform Test)

Algorithms that rely on geometric distance (K-Nearest Neighbors, SVMs, Logistic Regression, Neural Networks) or gradient descent are heavily distorted if one feature spans $[0, 100]$ and another spans $[0, 10000]$.

- **`StandardScaler`**: $z = \frac{x - \mu}{\sigma}$ (centers at 0, unit variance).
- **`RobustScaler`**: Uses median and IQR, resistant to residual outliers.


In [ ]:
# TODO: Scale continuous features ('tenure', 'MonthlyCharges', 'TotalCharges_log', 'charge_tenure_ratio') using StandardScaler
# Fit on Train, transform Train and Test
# YOUR CODE HERE


## 5. Assembling the Full Engineered Matrix

We now combine our continuous, interaction, frequency, target-encoded, and one-hot features into full modeling matrices.


In [ ]:
# TODO: Combine engineered numeric features with one-hot encoded features and save to ../data/
# X_train_final.to_csv('../data/telco_X_train_engineered.csv', index=False)
# X_test_final.to_csv('../data/telco_X_test_engineered.csv', index=False)
# YOUR CODE HERE
